# LeetCode 4042 — Valid K-Unique Subarrays II (Hard)

Bài toán LeetCode (bài premium):

```python
class Solution:
    def validSubarrays(self, nums: List[int], k: int, l0: int, r0: int, q: int) -> List[bool]:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4042

Mỗi truy vấn hỏi một đoạn có thỏa **hai** điều kiện hay không; truy vấn sau **phụ thuộc** đáp án truy vấn trước (truy vấn **online**), nên không thể sắp xếp lại truy vấn (không dùng được Mo's algorithm). Ta cần trả lời mỗi truy vấn trong `O(1)` sau khi tiền xử lý:

- "Đúng `k` giá trị phân biệt" → **cửa sổ trượt** tiền xử lý hai mảng `A[r]`, `B[r]`: đoạn `[l, r]` có đúng `k` giá trị ⇔ `A[r] <= l < B[r]`.
- "Mọi giá trị xuất hiện **chẵn** lần" → **XOR tiền tố** của "dấu vân tay" từng giá trị: bitmask chính xác, rồi thu gọn thành **băm ngẫu nhiên 64 bit** (Zobrist hashing).

---

## Sau notebook này, bạn sẽ nắm được

1. Tính đơn điệu của số giá trị phân biệt theo đầu trái.
2. Cửa sổ trượt tìm "đầu trái nhỏ nhất có đúng `kk` giá trị phân biệt" cho mọi đầu phải.
3. Kiểm tra "mọi số đếm chẵn" bằng XOR tiền tố: bitmask chính xác và băm ngẫu nhiên.
4. Cách sinh truy vấn online.
5. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng `nums` và số `k`. Đoạn `nums[l..r]` là **hợp lệ** nếu:

1. có **đúng `k`** giá trị phân biệt;
2. **mỗi** giá trị trong đoạn xuất hiện một số **chẵn** lần.

Có `q` truy vấn. Truy vấn đầu là `(l0, r0)`. Sau khi trả lời truy vấn `(l, r)` với kết quả `ans`:

```text
g = l + r      nếu ans == True
g = r - l      nếu ans == False
l, r = (l XOR g) % n, (r XOR g) % n
nếu l > r thì đổi chỗ l, r
```

Trả về danh sách `q` kết quả.

---

## Ví dụ 1

```python
nums = [1, 2, 2, 1], k = 2, l0 = 1, r0 = 2, q = 2   ->   [False, True]
```

- Truy vấn 1: `[2, 2]` chỉ có 1 giá trị → `False`. `g = 2 - 1 = 1` → `l = 1^1 = 0`, `r = 2^1 = 3`.
- Truy vấn 2: `[1, 2, 2, 1]`: 2 giá trị, mỗi giá trị xuất hiện 2 lần → `True`.

## Ví dụ 2

```python
nums = [1, 2, 3, 3, 4], k = 1, l0 = 2, r0 = 3, q = 2   ->   [True, False]
```

- Truy vấn 1: `[3, 3]` → `True`. `g = 2 + 3 = 5` → `l = (2^5) % 5 = 2`, `r = (3^5) % 5 = 1` → đổi chỗ thành `(1, 2)`.
- Truy vấn 2: `[2, 3]` có 2 giá trị → `False`.

---

## Ràng buộc

```text
1 <= n, q <= 5 * 10^5
1 <= nums[i] <= 10^9
1 <= k <= n
0 <= l0 <= r0 < n
```

# 2. Kiến thức nền tảng

## 2.1. Sinh truy vấn online

In [1]:
from typing import List
from collections import Counter


def next_query(l, r, ans, n):
    g = l + r if ans else r - l
    l, r = (l ^ g) % n, (r ^ g) % n
    if l > r:
        l, r = r, l
    return l, r


print(next_query(1, 2, False, 4), next_query(2, 3, True, 5))

(0, 3) (1, 2)


## 2.2. Đếm giá trị phân biệt khi đầu trái tăng

Cố định `r`. Khi `l` tăng (đoạn ngắn lại), số giá trị phân biệt **không tăng**, và mỗi bước giảm **tối đa 1**.

In [2]:
nums = [3, 1, 3, 2, 2]
r = 4
print([len(set(nums[l:r + 1])) for l in range(r + 1)])

[3, 3, 2, 1, 1]


Vì vậy các `l` cho đúng `k` giá trị phân biệt tạo thành **một đoạn liên tiếp** `[A[r], B[r])`. Ở ví dụ trên: với `k = 3` là `l ∈ {0, 1}` (đoạn `[0, 2)`); với `k = 2` chỉ có `l = 2` (đoạn `[2, 3)`); với `k = 1` là `l ∈ {3, 4}`.

## 2.3. XOR tiền tố và "xuất hiện chẵn lần"

Gán mỗi giá trị một "dấu vân tay" `h(x)`. Đặt `pre[t] = h(nums[0]) ^ ... ^ h(nums[t-1])`. Khi đó `pre[r+1] ^ pre[l]` = XOR các dấu vân tay trong đoạn; giá trị xuất hiện chẵn lần **tự triệt tiêu**.

- Nếu `h(x) = 1 << id(x)` (mỗi giá trị một bit riêng) → `pre[r+1] == pre[l]` **⇔** mọi giá trị chẵn lần (chính xác tuyệt đối).
- Nếu `h(x)` là số ngẫu nhiên 64 bit → đúng **gần như chắc chắn** (xác suất sai cỡ `2^-64` mỗi truy vấn).

In [3]:
nums = [1, 2, 2, 1, 3]
ids = {x: i for i, x in enumerate(dict.fromkeys(nums))}
pre = [0]
for x in nums:
    pre.append(pre[-1] ^ (1 << ids[x]))
print([bin(p) for p in pre])
print("đoạn [0..3] chẵn hết?", pre[4] == pre[0], "| đoạn [1..4]?", pre[5] == pre[1])

['0b0', '0b1', '0b11', '0b1', '0b0', '0b100']
đoạn [0..3] chẵn hết? True | đoạn [1..4]? False


# 3. Cách 1 — Brute Force: đếm từng truy vấn

Mỗi truy vấn tạo `Counter` của đoạn: `O(n)` → tổng `O(n · q)` — tới `2.5·10^11` với giới hạn đề.

In [4]:
def valid_bruteforce(nums: List[int], k: int, l: int, r: int, q: int) -> List[bool]:
    n = len(nums)
    res = []
    for _ in range(q):
        cnt = Counter(nums[l:r + 1])
        ok = len(cnt) == k and all(v % 2 == 0 for v in cnt.values())
        res.append(ok)
        l, r = next_query(l, r, ok, n)
    return res


print(valid_bruteforce([1, 2, 2, 1], 2, 1, 2, 2), valid_bruteforce([1, 2, 3, 3, 4], 1, 2, 3, 2))

[False, True] [True, False]


# 4. Điều kiện 1 bằng cửa sổ trượt: `A[r]` và `B[r]`

Định nghĩa `min_left(kk)[r]` = đầu trái **nhỏ nhất** sao cho `nums[left..r]` có **đúng** `kk` giá trị phân biệt (hoặc `n` nếu không có).

Cửa sổ trượt chuẩn: tăng `r`, thêm `nums[r]`; khi số giá trị phân biệt `> kk` thì tăng `left`. Sau đó `left` là đầu trái nhỏ nhất có `<= kk` giá trị; nếu số giá trị đúng bằng `kk` thì ghi nhận.

- `A = min_left(k)`: `l >= A[r]` ⇔ đoạn có `<= k` giá trị (và `A[r] < n` nghĩa là có thể đạt đúng `k`).
- `B = min_left(k - 1)`: `l < B[r]` ⇔ đoạn có `>= k` giá trị. Với `k = 1` thì `B[r] = r + 1` (mọi `l <= r` đều có `>= 1` giá trị).

## Mô phỏng ví dụ 1: `nums = [1, 2, 2, 1]`, `k = 2`

| r | đoạn dài nhất có `<= 2` gtpb | A[r] | đoạn dài nhất có `<= 1` gtpb | B[r] | các `l` hợp lệ |
|---:|---|---:|---|---:|---|
| 0 | `[1]` (chỉ 1 gtpb) | 4 (= n) | `[1]` | 0 | không có |
| 1 | `[1, 2]` | 0 | `[2]` | 1 | `{0}` |
| 2 | `[1, 2, 2]` | 0 | `[2, 2]` | 1 | `{0}` |
| 3 | `[1, 2, 2, 1]` | 0 | `[1]` | 3 | `{0, 1, 2}` |

(gtpb = giá trị phân biệt.) Truy vấn `(1, 2)`: `A[2] = 0 <= 1` nhưng `1 < B[2] = 1` sai → không đúng 2 giá trị ✔. Truy vấn `(0, 3)`: `0 <= 0 < 3` ✔.

In [5]:
def min_left(nums: List[int], kk: int) -> List[int]:
    n = len(nums)
    res = [n] * n
    if kk == 0:
        return res
    cnt = {}
    left = 0
    for right, x in enumerate(nums):
        cnt[x] = cnt.get(x, 0) + 1
        while len(cnt) > kk:
            y = nums[left]
            cnt[y] -= 1
            if cnt[y] == 0:
                del cnt[y]
            left += 1
        if len(cnt) == kk:
            res[right] = left
    return res


def bounds(nums, k):
    A = min_left(nums, k)
    B = min_left(nums, k - 1) if k > 1 else list(range(1, len(nums) + 1))
    return A, B


A, B = bounds([1, 2, 2, 1], 2)
print("A =", A, "| B =", B)

A = [4, 0, 0, 0] | B = [0, 1, 1, 3]


In [6]:
# Kiểm tra: A[r] <= l < B[r]  <=>  đúng k giá trị phân biệt
import random

random.seed(42)
for _ in range(2000):
    arr = [random.randint(1, 4) for _ in range(random.randint(1, 9))]
    kk = random.randint(1, len(arr))
    A, B = bounds(arr, kk)
    for r in range(len(arr)):
        for l in range(r + 1):
            assert (A[r] <= l < B[r]) == (len(set(arr[l:r + 1])) == kk), (arr, kk, l, r)
print("Điều kiện A[r] <= l < B[r] đúng trên 2000 mảng ngẫu nhiên")

Điều kiện A[r] <= l < B[r] đúng trên 2000 mảng ngẫu nhiên


# 5. Cách 2 — Bitmask chẵn lẻ chính xác

Kết hợp: điều kiện 1 bằng `A`, `B`; điều kiện 2 bằng `pre` dùng bitmask `1 << id(x)` (số nguyên lớn của Python). Mỗi truy vấn `O(1)` so sánh... nhưng so sánh hai số nguyên `D` bit tốn `O(D / 64)`, và **bộ nhớ** `O(n · D / 64)` (`D` = số giá trị phân biệt) — với `n = D = 5·10^5` là cỡ `4·10^9` từ máy: không khả thi. Cách này để **hiểu** và để **kiểm chứng**.

In [7]:
def valid_bitmask(nums: List[int], k: int, l: int, r: int, q: int) -> List[bool]:
    n = len(nums)
    A, B = bounds(nums, k)
    ids = {}
    pre = [0]
    for x in nums:
        if x not in ids:
            ids[x] = len(ids)
        pre.append(pre[-1] ^ (1 << ids[x]))
    res = []
    for _ in range(q):
        ok = A[r] <= l < B[r] and pre[r + 1] == pre[l]
        res.append(ok)
        l, r = next_query(l, r, ok, n)
    return res


print(valid_bitmask([1, 2, 2, 1], 2, 1, 2, 2), valid_bitmask([1, 2, 3, 3, 4], 1, 2, 3, 2))

[False, True] [True, False]


# 6. Cách 3 — Băm ngẫu nhiên 64 bit (Zobrist hashing)

Thay `1 << id(x)` bằng một số **ngẫu nhiên 64 bit** `h(x)`. Mỗi `pre[t]` chỉ còn một số 64 bit → bộ nhớ `O(n)`, so sánh `O(1)`.

- Nếu mọi giá trị chẵn lần: XOR = 0 → `pre[r+1] == pre[l]` (luôn đúng).
- Nếu có giá trị lẻ lần: XOR là XOR của một tập **khác rỗng** các số ngẫu nhiên độc lập → bằng 0 với xác suất `2^-64`.

Thêm một bộ lọc rẻ: độ dài đoạn phải **chẵn** (tổng các số chẵn là số chẵn).

Trong thi đấu, nên lấy hạt giống ngẫu nhiên thật (ví dụ `random.getrandbits` không cố định seed) để không ai dựng được test phá. Ở đây cố định seed để notebook tái lập được.

## Mô phỏng ví dụ 2: `nums = [1, 2, 3, 3, 4]`, `k = 1`

| truy vấn | (l, r) | đoạn | `A[r] <= l < B[r]` | chẵn hết | kết quả | (l, r) tiếp |
|---:|---|---|:---:|:---:|:---:|---|
| 1 | (2, 3) | `[3, 3]` | ✔ (`2 <= 2 < 4`) | ✔ | True | `g = 5` → (1, 2) |
| 2 | (1, 2) | `[2, 3]` | ✘ (`A[2] = 2 > 1`) | ✘ | False | |

In [8]:
def valid_hash(nums: List[int], k: int, l: int, r: int, q: int, seed: int = 12345, verbose: bool = False) -> List[bool]:
    n = len(nums)
    A, B = bounds(nums, k)
    rnd = random.Random(seed)
    h = {}
    pre = [0]
    for x in nums:
        if x not in h:
            h[x] = rnd.getrandbits(64)
        pre.append(pre[-1] ^ h[x])
    if verbose:
        print("A =", A, "| B =", B)
    res = []
    for _ in range(q):
        ok = (r - l + 1) % 2 == 0 and pre[r + 1] == pre[l] and A[r] <= l < B[r]
        if verbose:
            print(f"(l, r) = ({l}, {r}) đoạn {nums[l:r + 1]} -> {ok}")
        res.append(ok)
        l, r = next_query(l, r, ok, n)
    return res


print(valid_hash([1, 2, 3, 3, 4], 1, 2, 3, 2, verbose=True))

A = [0, 1, 2, 2, 4] | B = [1, 2, 3, 4, 5]
(l, r) = (2, 3) đoạn [3, 3] -> True
(l, r) = (1, 2) đoạn [2, 3] -> False
[True, False]


# 7. Phiên bản đúng format LeetCode

In [9]:
import random
from typing import List


class Solution:
    def validSubarrays(self, nums: List[int], k: int, l0: int, r0: int, q: int) -> List[bool]:
        n = len(nums)

        def min_left(kk):
            res = [n] * n
            if kk == 0:
                return res
            cnt = {}
            left = 0
            for right, x in enumerate(nums):
                cnt[x] = cnt.get(x, 0) + 1
                while len(cnt) > kk:
                    y = nums[left]
                    cnt[y] -= 1
                    if cnt[y] == 0:
                        del cnt[y]
                    left += 1
                if len(cnt) == kk:
                    res[right] = left
            return res

        A = min_left(k)
        B = min_left(k - 1) if k > 1 else list(range(1, n + 1))

        rnd = random.Random(20261119)
        h = {}
        pre = [0] * (n + 1)
        for i, x in enumerate(nums):
            if x not in h:
                h[x] = rnd.getrandbits(64)
            pre[i + 1] = pre[i] ^ h[x]

        res = []
        l, r = l0, r0
        for _ in range(q):
            ok = (r - l + 1) % 2 == 0 and pre[r + 1] == pre[l] and A[r] <= l < B[r]
            res.append(ok)
            g = l + r if ok else r - l
            l, r = (l ^ g) % n, (r ^ g) % n
            if l > r:
                l, r = r, l
        return res

In [10]:
sol = Solution()
tests = [
    ([1, 2, 2, 1], 2, 1, 2, 2, [False, True], "ví dụ 1"),
    ([1, 2, 3, 3, 4], 1, 2, 3, 2, [True, False], "ví dụ 2"),
    ([5], 1, 0, 0, 3, [False, False, False], "n = 1, độ dài lẻ"),
    ([7, 7], 1, 0, 1, 2, [True, True], "[7, 7] lặp lại"),
    ([1, 1, 2, 2], 1, 0, 3, 1, [False], "chẵn hết nhưng 2 giá trị"),
]
for nums, k, l0, r0, q, expected, name in tests:
    result = sol.validSubarrays(nums, k, l0, r0, q)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:26} -> {result}")

PASS | ví dụ 1                    -> [False, True]
PASS | ví dụ 2                    -> [True, False]
PASS | n = 1, độ dài lẻ           -> [False, False, False]
PASS | [7, 7] lặp lại             -> [True, True]
PASS | chẵn hết nhưng 2 giá trị   -> [False]


Test `[7, 7]`: truy vấn 1 `(0, 1)` → `True`, `g = 1` → `l = 1 % 2 = 1`, `r = 0` → đổi chỗ `(0, 1)` → lại `True`.

# 8. Vì sao đúng?

1. **Điều kiện 1:** với `r` cố định, `f(l) = số giá trị phân biệt của nums[l..r]` không tăng và giảm từng đơn vị (từ `f(0)` xuống `f(r) = 1`). Cửa sổ trượt cho `A[r]` = `l` nhỏ nhất với `f(l) <= k` (và `f(A[r]) = k` nếu `A[r] < n`), `B[r]` = `l` nhỏ nhất với `f(l) <= k-1`. Vậy `f(l) = k` ⇔ `A[r] <= l < B[r]`. (Nếu `f(0) < k` thì `A[r] = n`, không `l` nào thỏa — đúng.)
2. **Điều kiện 2:** XOR các dấu vân tay trong đoạn = XOR các `h(x)` với `x` xuất hiện **lẻ** lần. Bằng 0 khi không có `x` nào lẻ; ngược lại khác 0 (chắc chắn với bitmask, gần như chắc chắn với băm 64 bit).
3. **Online:** mỗi truy vấn `O(1)`, ta mô phỏng đúng công thức sinh truy vấn.

# 9. Độ phức tạp

| Cách | Ý tưởng | Tiền xử lý | Mỗi truy vấn | Bộ nhớ |
|---|---|---:|---:|---:|
| 1. Brute Force | `Counter` mỗi đoạn | — | `O(n)` | `O(n)` |
| 2. Bitmask | `A`, `B` + XOR bitmask | `O(n · D / 64)` | `O(D / 64)` | `O(n · D / 64)` |
| 3. Băm 64 bit | `A`, `B` + XOR băm | `O(n)` | `O(1)` | `O(n)` |

In [11]:
import time

random.seed(4042)
n = 2 * 10**5
big = [random.randint(1, 3) for _ in range(n)]
start = time.time()
out = sol.validSubarrays(big, 2, 0, 99, 2 * 10**5)
print("n = q = 200000:", f"{time.time() - start:.2f} giây;", len(out), "kết quả")

n = q = 200000: 0.17 giây; 200000 kết quả


# 10. Đối chiếu ba cách

In [12]:
random.seed(7)
for _ in range(3000):
    n = random.randint(1, 10)
    arr = [random.randint(1, 3) for _ in range(n)]
    kk = random.randint(1, n)
    l = random.randint(0, n - 1)
    r = random.randint(l, n - 1)
    qq = random.randint(1, 8)
    expected = valid_bruteforce(arr, kk, l, r, qq)
    assert valid_bitmask(arr, kk, l, r, qq) == expected
    assert valid_hash(arr, kk, l, r, qq) == expected
    assert sol.validSubarrays(arr, kk, l, r, qq) == expected
print("Ba cách khớp brute force trên 3000 test")

Ba cách khớp brute force trên 3000 test


# 11. Những lỗi dễ mắc

## Lỗi 1 — Chỉ kiểm tra độ dài chẵn

Độ dài chẵn là điều kiện **cần**, không đủ: `[1, 2]` dài 2 nhưng mỗi giá trị xuất hiện 1 lần.

In [13]:
def wrong_length_only(nums, k, l, r):
    return (r - l + 1) % 2 == 0 and len(set(nums[l:r + 1])) == k


print("Sai :", wrong_length_only([1, 2], 2, 0, 1), "| Đúng:", valid_bruteforce([1, 2], 2, 0, 1, 1)[0])

Sai : True | Đúng: False


## Lỗi 2 — Dùng tổng tiền tố thay vì XOR

"Tổng `h(x)` trong đoạn = 0" không có nghĩa gì về chẵn lẻ; còn so `sum` của các giá trị thì `[1, 3]` và `[2, 2]` giống nhau. XOR mới có tính chất `a ^ a = 0`.

In [14]:
def wrong_sum_parity(nums, l, r):
    pre = [0]
    for x in nums:
        pre.append(pre[-1] + x)
    # SAI: coi "tổng đoạn chẵn" là "mọi giá trị xuất hiện chẵn lần"
    return (pre[r + 1] - pre[l]) % 2 == 0


print("Sai :", wrong_sum_parity([1, 3], 0, 1), "| Đúng: mỗi giá trị chẵn lần?", all(v % 2 == 0 for v in Counter([1, 3]).values()))

Sai : True | Đúng: mỗi giá trị chẵn lần? False


## Lỗi 3 — Dùng `h(x) = x` (không ngẫu nhiên)

XOR các giá trị gốc có thể bằng 0 dù có giá trị lẻ lần: `1 ^ 2 ^ 3 = 0`.

In [15]:
def wrong_identity_hash(nums, k, l, r):
    pre = [0]
    for x in nums:
        pre.append(pre[-1] ^ x)                 # SAI: h(x) = x
    return pre[r + 1] == pre[l] and len(set(nums[l:r + 1])) == k


arr = [1, 2, 3, 1, 2, 3]
print("XOR của [1, 2, 3] =", 1 ^ 2 ^ 3)
print("h(x) = x :", wrong_identity_hash(arr, 3, 0, 2), "| Đúng:", valid_bruteforce(arr, 3, 0, 2, 1)[0])

XOR của [1, 2, 3] = 0
h(x) = x : True | Đúng: False


## Lỗi 4 — Sai `B` khi `k = 1`

Với `k = 1`, điều kiện "có `>= 1` giá trị phân biệt" luôn đúng với đoạn khác rỗng, nên `B[r]` phải là `r + 1` (hoặc `n` — cũng được vì luôn có `l <= r`). Lỗi hay gặp: nghĩ "đoạn có `<= 0` giá trị bắt đầu ở 0" rồi gán `B[r] = 0` → `l < 0` luôn sai → **mọi** truy vấn với `k = 1` thành `False`.

In [16]:
def wrong_k1(nums, k, l, r, q):
    n = len(nums)
    A = min_left(nums, k)
    B = min_left(nums, k - 1) if k > 1 else [0] * n        # SAI: coi "<= 0 giá trị" bắt đầu ở 0
    res = []
    for _ in range(q):
        ok = A[r] <= l < B[r] and all(v % 2 == 0 for v in Counter(nums[l:r + 1]).values())
        res.append(ok)
        l, r = next_query(l, r, ok, n)
    return res


print("Sai :", wrong_k1([1, 2, 3, 3, 4], 1, 2, 3, 2), "| Đúng:", sol.validSubarrays([1, 2, 3, 3, 4], 1, 2, 3, 2))

Sai : [False, False] | Đúng: [True, False]


## Lỗi 5 — Sinh truy vấn sai thứ tự

Phải tính `g` từ `(l, r)` **hiện tại** và đáp án **hiện tại**, rồi cập nhật **cả hai** `l, r` cùng lúc (`l, r = ...`), cuối cùng mới đổi chỗ nếu `l > r`. Cập nhật `l` trước rồi dùng `l` mới để tính `r` là sai.

In [17]:
def wrong_query_update(l, r, ans, n):
    g = l + r if ans else r - l
    l = (l ^ g) % n
    g = l + r if ans else r - l            # SAI: tính lại g bằng l mới
    r = (r ^ g) % n
    if l > r:
        l, r = r, l
    return l, r


print("Sai :", wrong_query_update(0, 1, True, 2), "| Đúng:", next_query(0, 1, True, 2))

Sai : (1, 1) | Đúng: (0, 1)


# 12. Mẹo nhận diện

1. "Truy vấn sau phụ thuộc đáp án truy vấn trước" → **online**: phải tiền xử lý để trả lời mỗi truy vấn nhanh (không Mo's, không sắp xếp truy vấn).
2. "Đúng `k` giá trị phân biệt" → "`<= k`" trừ "`<= k-1`" — kỹ thuật kinh điển của *Subarrays with K Different Integers*.
3. "Mọi giá trị xuất hiện chẵn lần" → **XOR tiền tố của băm ngẫu nhiên** (Zobrist). Bitmask chính xác khi số giá trị nhỏ (≤ 64 hoặc ≤ vài chục).

Các bài liên quan: *Subarrays with K Different Integers* (LeetCode 992), *Find the Longest Substring Containing Vowels in Even Counts* (1371 — bitmask 5 bit), *Number of Wonderful Substrings* (1915).

# 13. Biến thể

## Biến thể 1 — Đếm tất cả đoạn hợp lệ (offline)

Với mỗi `r`, đếm `l ∈ [A[r], B[r])` có `pre[l] == pre[r+1]`: lưu danh sách vị trí của từng giá trị `pre` rồi `bisect`. (Bài tập 6.)

## Biến thể 2 — "Mọi giá trị xuất hiện bội của 3 lần"

Thay XOR bằng cộng **modulo 3** theo từng "chữ số" — hoặc băm: vector đếm mod 3 nhân hệ số ngẫu nhiên, lấy modulo một số nguyên tố lớn.

# 14. Template tư duy

```python
# Đúng k giá trị phân biệt: A = min_left(k), B = min_left(k-1)
# Mọi số đếm chẵn: pre[t] = XOR h(nums[0..t-1]), h ngẫu nhiên 64 bit
ok = (r - l + 1) % 2 == 0 and pre[r + 1] == pre[l] and A[r] <= l < B[r]
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`nums = [3, 1, 3, 1, 2]`, `k = 2`. Tính `A` và `B`. Những đoạn nào kết thúc tại `r = 3` là hợp lệ? Kiểm tra bằng cell dưới.

In [18]:
# Kiểm tra Bài 1
nums = [3, 1, 3, 1, 2]
A, B = bounds(nums, 2)
print("A =", A, "| B =", B)
print([(l, 3) for l in range(4) if A[3] <= l < B[3] and all(v % 2 == 0 for v in Counter(nums[l:4]).values())])

A = [5, 0, 0, 0, 3] | B = [0, 1, 2, 3, 4]
[(0, 3)]


## Bài 2 — Dự đoán output

```python
print(min_left([1, 1, 2, 3], 2))
print(next_query(0, 3, False, 4))
print(valid_bruteforce([4, 4, 4, 4], 1, 0, 2, 2))
```

In [19]:
print(min_left([1, 1, 2, 3], 2))
print(next_query(0, 3, False, 4))
print(valid_bruteforce([4, 4, 4, 4], 1, 0, 2, 2))

[4, 4, 0, 2]
(0, 3)
[False, False]


## Bài 3 — Hoàn thiện `min_left`

In [20]:
def exercise_3_min_left(nums: List[int], kk: int) -> List[int]:
    # TODO: cửa sổ trượt, ghi đầu trái nhỏ nhất có đúng kk giá trị phân biệt (mặc định n)
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_min_left([1, 2, 2, 1], 2) == [4, 0, 0, 0]

## Bài 4 — Subarrays with K Different Integers (LeetCode 992)

Đếm số đoạn có đúng `k` giá trị phân biệt, `O(n)`.

```python
subarrays_with_k_distinct([1, 2, 1, 2, 3], 2)   -> 7
```

In [21]:
def exercise_4_k_distinct(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_k_distinct([1, 2, 1, 2, 3], 2) == 7

## Bài 5 — Đếm đoạn mà mọi giá trị xuất hiện chẵn lần

Dùng XOR tiền tố băm + `Counter` các giá trị `pre`, `O(n)`.

```python
count_even_subarrays([1, 2, 2, 1, 1])   -> 4     # [2, 2], [1, 1], [1, 2, 2, 1], [2, 2, 1, 1]
```

In [22]:
def exercise_5_count_even(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError

## Bài 6 — Mini challenge: đếm tất cả đoạn hợp lệ

Đếm số đoạn có đúng `k` giá trị phân biệt và mọi giá trị chẵn lần, `O(n log n)`.

In [23]:
def exercise_6_count_valid(nums: List[int], k: int) -> int:
    # TODO
    raise NotImplementedError

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `cnt` là dict đếm trong cửa sổ.
- Mức 2: khi `len(cnt) > kk`, bỏ `nums[left]` và tăng `left`.
- Mức 3: ghi `res[right] = left` khi `len(cnt) == kk`.

**Bài 4**

- Mức 1: với mỗi `r`, số `l` hợp lệ là `B[r] - A[r]` (nếu `A[r] < n`).
- Mức 2: dùng `bounds`.
- Mức 3: hoặc `atMost(k) - atMost(k - 1)`.

**Bài 5**

- Mức 1: `seen = Counter({0: 1})`.
- Mức 2: tại mỗi tiền tố `p`, cộng `seen[p]` rồi `seen[p] += 1`.
- Mức 3: dùng băm ngẫu nhiên cho từng giá trị.

**Bài 6**

- Mức 1: `pos[p]` = danh sách chỉ số `t` (tăng dần) có `pre[t] == p`.
- Mức 2: với mỗi `r`, đếm `t ∈ [A[r], B[r])` trong `pos[pre[r+1]]` bằng `bisect`.
- Mức 3: chỉ xét khi `A[r] < n`.

# 16. Đáp án tham khảo

## Đáp án Bài 1

`nums = [3, 1, 3, 1, 2]`, `k = 2`:

- `A = [5, 0, 0, 0, 3]`: `r = 0` chỉ có 1 giá trị (→ `n = 5`); `r = 1, 2, 3`: cả tiền tố chỉ có `{3, 1}`; `r = 4`: đoạn dài nhất kết thúc ở `2` có `<= 2` giá trị là `[1, 2]` (bắt đầu ở 3).
- `B = [0, 1, 2, 3, 4]`: đoạn dài nhất có 1 giá trị kết thúc tại `r` chỉ là chính `nums[r]` (không có hai số liền nhau bằng nhau).
- Tại `r = 3`, các `l` có đúng 2 giá trị: `l ∈ [0, 3)`. `l = 0`: `[3, 1, 3, 1]` — mỗi giá trị 2 lần ✔; `l = 1`: `[1, 3, 1]` ✘; `l = 2`: `[3, 1]` ✘. Đáp số: chỉ `(0, 3)`.

## Đáp án Bài 2

```text
[4, 4, 0, 2]
(0, 3)
[False, False]
```

- `min_left([1, 1, 2, 3], 2)`: `r = 0, 1` chỉ có giá trị `1` → `4`; `r = 2`: `[1, 1, 2]` → `0`; `r = 3`: `[2, 3]` → `2`.
- `next_query(0, 3, False, 4)`: `g = 3`, `l = 3`, `r = 0` → đổi chỗ `(0, 3)`.
- `[4, 4, 4]` lẻ → `False`; `g = 2` → `l = 2`, `r = 0` → `(0, 2)` lại → `False`.

In [24]:
# Đáp án Bài 3
def answer_3_min_left(nums: List[int], kk: int) -> List[int]:
    return min_left(nums, kk)


assert answer_3_min_left([1, 2, 2, 1], 2) == [4, 0, 0, 0]
print("Bài 3: PASS")

Bài 3: PASS


In [25]:
# Đáp án Bài 4
def answer_4_k_distinct(nums: List[int], k: int) -> int:
    n = len(nums)
    A, B = bounds(nums, k)
    return sum(B[r] - A[r] for r in range(n) if A[r] < n)


assert answer_4_k_distinct([1, 2, 1, 2, 3], 2) == 7
assert answer_4_k_distinct([1, 2, 1, 3, 4], 3) == 3
print("Bài 4: PASS")

Bài 4: PASS


In [26]:
# Đáp án Bài 5
def answer_5_count_even(nums: List[int]) -> int:
    rnd = random.Random(5)
    h = {}
    seen = Counter({0: 1})
    p = res = 0
    for x in nums:
        if x not in h:
            h[x] = rnd.getrandbits(64)
        p ^= h[x]
        res += seen[p]
        seen[p] += 1
    return res


def brute_even(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if all(v % 2 == 0 for v in Counter(nums[i:j + 1]).values()))


print(answer_5_count_even([1, 2, 2, 1, 1]), brute_even([1, 2, 2, 1, 1]))
random.seed(55)
for _ in range(500):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 10))]
    assert answer_5_count_even(arr) == brute_even(arr), arr
print("Bài 5: PASS")

4 4
Bài 5: PASS


In [27]:
# Đáp án Bài 6
from bisect import bisect_left
from collections import defaultdict


def answer_6_count_valid(nums: List[int], k: int) -> int:
    n = len(nums)
    A, B = bounds(nums, k)
    rnd = random.Random(6)
    h = {}
    pre = [0]
    for x in nums:
        if x not in h:
            h[x] = rnd.getrandbits(64)
        pre.append(pre[-1] ^ h[x])
    pos = defaultdict(list)
    for t, p in enumerate(pre):
        pos[p].append(t)
    total = 0
    for r in range(n):
        if A[r] < n:
            lst = pos[pre[r + 1]]
            total += bisect_left(lst, B[r]) - bisect_left(lst, A[r])
    return total


def brute_valid_count(nums, k):
    n = len(nums)
    cnt = 0
    for i in range(n):
        for j in range(i, n):
            c = Counter(nums[i:j + 1])
            if len(c) == k and all(v % 2 == 0 for v in c.values()):
                cnt += 1
    return cnt


random.seed(6)
for _ in range(1000):
    arr = [random.randint(1, 3) for _ in range(random.randint(1, 10))]
    kk = random.randint(1, 3)
    assert answer_6_count_valid(arr, kk) == brute_valid_count(arr, kk), (arr, kk)
print("Bài 6: PASS (khớp brute force)")

Bài 6: PASS (khớp brute force)


# 17. Tổng kết

**Valid K-Unique Subarrays II**:

1. Truy vấn online → tiền xử lý để mỗi truy vấn `O(1)`.
2. Đúng `k` giá trị phân biệt ⇔ `A[r] <= l < B[r]` với `A = min_left(k)`, `B = min_left(k - 1)` (cửa sổ trượt).
3. Mọi giá trị chẵn lần ⇔ `pre[r+1] == pre[l]` với XOR tiền tố của băm ngẫu nhiên 64 bit (bitmask là phiên bản chính xác nhưng tốn bộ nhớ).

## Độ phức tạp

```text
Time:  O(n + q)
Space: O(n)
```

# 18. Checklist trước khi submit

- [ ] Dùng **XOR** (không phải tổng) và băm **ngẫu nhiên** (không phải `h(x) = x`)?
- [ ] `B[r] = r + 1` khi `k = 1`?
- [ ] Sinh truy vấn: tính `g` từ `(l, r, ans)` hiện tại, cập nhật đồng thời, rồi đổi chỗ nếu `l > r`?
- [ ] Có bộ lọc độ dài chẵn (rẻ, loại sớm)?
- [ ] Đã test: `n = 1`, `k = 1`, chẵn hết nhưng sai số giá trị phân biệt?